Week 2 — Day 1: Data Engineering Fundamentals
Part 1 — Data Engineering and Source Systems

Q. 1 — Build a Source Inventory

Answer:
Source	                     Type of data	               How often it may change	                                One check before using it
MySQL orders	                structured                  often (in real time or throughout the day)              Check row count and required columns 
Supplier CSV	                structured                  occasionally (weekly or monthly)                        Check header and data types
Website JSON	                semi-structured             often (whenever the website data is updated)            Check expected field          
Excel stock report	            structured                  Daily (or whenever stock levels are updated)             Check sheet, columns, and missing values

Q. 2 — Decide What the Data Engineer Must Do

Answer:
A. Build a daily process that copies orders into analytical storage. -  Data engineering
B. Create a chart showing sales by city. - Data analytics
C. Train a model to predict customer churn. - Data science
D. Remove duplicate order records before reporting. - Data engineering
E. Make sure a failed pipeline can be rerun safely. - Data engineering

Q. 3 — Inspect Three Small Input Batches
Create three DataFrames and print:
- row count for each batch;
- total rows after combining them;
- total sales after combining them.

In [18]:
import pandas as pd

batch_1 = pd.DataFrame(
    {
        "order_id": [101, 102],
        "city": ["London", "Manchester"],
        "amount_gbp": [45.50, 18.00]
    }
)
print(batch_1)
print("Batch 1:", len(batch_1), "rows")

   order_id        city  amount_gbp
0       101      London        45.5
1       102  Manchester        18.0
Batch 1: 2 rows


In [17]:
import pandas as pd

batch_2 = pd.DataFrame(
    {
        "order_id": [103, 104],
        "city": ["Bristol", "London"],
        "amount_gbp": [62.25, 27.40]
    }
)
print(batch_2)
print("Batch 2:", len(batch_2), "rows")



   order_id     city  amount_gbp
0       103  Bristol       62.25
1       104   London       27.40
Batch 2: 2 rows


In [16]:
import pandas as pd

batch_3 = pd.DataFrame(
    {
        "order_id": [101, 102],
        "city": ["Leeds", "Manchester"],
        "amount_gbp": [34.80, 51.10]
    }
)
print(batch_3)
print("Batch 3:", len(batch_3), "rows")

   order_id        city  amount_gbp
0       101       Leeds        34.8
1       102  Manchester        51.1
Batch 3: 2 rows


In [20]:
all_orders = pd.concat([batch_1, batch_2, batch_3], ignore_index=True)

print("Total rows after combining:", len(all_orders), "rows")
print("Total sales after combining:", all_orders["amount_gbp"].sum().round(2))

Total rows after combining: 6 rows
Total sales after combining: 239.05


Q. 4 — MCQ: Data Engineering
Which task is most clearly a data-engineering responsibility?
A. Choosing colors for a sales dashboard B. Building a reliable process that moves and cleans order data C. Writing a business presentation about monthly revenue D. Designing a customer survey

Answer: A

Part 2 — Pipeline Lifecycle: Ingestion, Storage, Processing, and Serving
Q. 5 — Put the Pipeline Steps in Order

Read new order records - Store the received records - Clean invalid values - Create daily sales summary - Make the summary available to analysts - 
Check that the run completed

 


Q. 6 — Design a Small Pipeline for Daily Orders
BritMart receives one CSV file every night.

Complete this design:

Pipeline step	                        What happens?	                            Simple check
Source	                                Supplier sends order file	                    File exists
Ingestion	                            Read the file	                            Expected columns are present
Storage	                                Keep received data	                        Row count matches input
Processing	                            Clean and transform data	                Invalid rows are counted
Serving	                                Create useful sales output	                Business totals are checked
Monitoring	                            Record whether the run succeeded	        Status and counts are available



Q. 7 — Process Three Batches and Keep the Counts

In [23]:
batch_1["batch_id"]= "batch_1" 
batch_2["batch_id"]= "batch_2"
batch_3["batch_id"]= "batch_3"

all_orders = pd.concat([batch_1, batch_2, batch_3], ignore_index=True)
print(all_orders)

summary = (
    all_orders
    .groupby("batch_id", as_index=False)
    .agg(
        row_count=("order_id", "count"),
        sales_gbp=("amount_gbp", "sum"),
    )
)

print(summary)

   order_id        city  amount_gbp batch_id
0       101      London       45.50  batch_1
1       102  Manchester       18.00  batch_1
2       103     Bristol       62.25  batch_2
3       104      London       27.40  batch_2
4       101       Leeds       34.80  batch_3
5       102  Manchester       51.10  batch_3
  batch_id  row_count  sales_gbp
0  batch_1          2      63.50
1  batch_2          2      89.65
2  batch_3          2      85.90


Q. 8 — MCQ: Ingestion
What does ingestion mean in a data pipeline?

A. Moving data from a source into the data platform B. Creating the final dashboard colours C. Deleting every raw record after reading it D. Training a prediction model

Answer: A

Part 3 — Transformation, Batch Processing, and Streaming Processing
Q. 9 — Clean a Batch of Orders

Rules:

city must not be missing;
amount must be greater than 0;
trim spaces from valid city names.

In [25]:
orders = pd.DataFrame({
    "order_id": [201, 202, 203, 204],
    "city": [" London ", "Manchester", "Leeds", None],
    "amount_gbp": [45.50, -5.00, 30.00, 20.00],
})

# trim spaces from valid city names:

orders["city"] = orders["city"].str.strip()

# check rules
valid_data = orders["city"].notna() & (orders["amount_gbp"] > 0)

# valid and rejected orders:

valid_orders = orders[valid_data].copy()
rejected_orders = orders[~valid_data].copy()

print(valid_orders)
print(rejected_orders)



   order_id    city  amount_gbp
0       201  London        45.5
2       203   Leeds        30.0
   order_id        city  amount_gbp
1       202  Manchester        -5.0
3       204         NaN        20.0


Q. 10 — Batch or Streaming?

Requirement	                    Choice	                Reason
Daily supplier file	            Batch	                Processes a large amount of data at a scheduled time
Payment events within seconds	Streaming	            Processes payments immediately as events happen
Monthly finance report	        Batch	                Processes data at a specific time each month to create the report
Live website activity	        Streaming	            Processes website activity in real time

Q. 11 — Simulate Three Batch Runs
Use:
batch_1 = [10, 20, 30]
batch_2 = [40, 50]
batch_3 = [60, 70, 80, 90]

For each batch, print:
batch number
record count
batch total
cumulative total

In [26]:
batches = [
    [10, 20, 30],
    [40, 50],
    [60, 70, 80, 90]
]

cumulative_total = 0

for batch_number, batch in enumerate(batches, start=1):
    record_count = len(batch)
    batch_total = sum(batch)
    cumulative_total += batch_total

    print(
        batch_number,
        record_count,
        batch_total,
        cumulative_total
    )

1 3 60 60
2 2 90 150
3 4 300 450


In [30]:
batch_1 = [10, 20, 30]
batch_2 = [40, 50]
batch_3 = [60, 70, 80, 90]

batches = [batch_1, batch_2, batch_3]

batch_number = 1
cumulative_total = 0

for batch in batches:
    record_count = len(batch)
    batch_total = sum(batch)
    cumulative_total += batch_total

    print(batch_number, record_count, batch_total, cumulative_total)

    batch_number += 1

1 3 60 60
2 2 90 150
3 4 300 450


Q. 12 — MCQ: Streaming
Which requirement is the strongest reason to use streaming rather than a daily batch?

A. A report that is needed once per month B. A file that arrives only once each night C. Fraud events that must be checked within seconds D. A yearly archive export

Answer: C


Part 4 — OLTP, OLAP, ETL, ELT, and Platform Design

Q. 13 — OLTP or OLAP?
Classify each workload.

A. Record a new customer order. - OLTP
B. Update the payment status of one order. - OLTP
C. Calculate sales by city for the last 12 months. - OLAP
D. Compare average order value across product categories. - OLAP

Q. 14 — ETL or ELT?
BritMart has two possible designs.

Design A - ETL
Extract source data
Clean it before loading
Load only cleaned data to the target

Design B - ELT
Extract source data
Load the source data to the target platform
Transform it inside the target


Q. 15 — Draw a Small Modern Data Platform
Use only the concepts covered today.

Your design must show:

at least 3 source systems
ingestion
storage
processing
serving
monitoring

Answer:

    MySQL Orders 
    Supplier CSV Files -->   Ingestion --> Storage --> Processing --> Serving --> Monitoring
    
    Processing --> Monitoring
    Storage --> Monitoring
    Ingestion --> Monitoring
   

Q. 16 — MCQ: OLAP
Which workload is an OLAP workload?
A. Insert one new payment record B. Change one customer's phone number C. Cancel one order D. Calculate three years of sales by region

Answer: D


Optional Practice
Q. 17 — Design Recovery for a Failed Batch

batch_1 -> received and processed
batch_2 -> received but processing failed
batch_3 -> not started

Answer:

Check why batch_2 failed.
Check whether any data was already saved or processed.
Fix the problem that caused the failure.
Rerun batch_2 only, because batch_1 has already succeeded.
Check that the input and output counts match.
Continue with batch_3.
Record each batch status and counts so nothing is lost or processed twice.

Q. 18 — Compare Two Pipeline Designs
Which design gives better support for rerunning the cleaning step if business rules change? Explain why.

Source -> Clean -> Final report
Source -> Store received data -> Clean -> Final report

Answer: Design B stores the received data before cleaning it



Q. 19 — Reconcile Three Batches

Expected source counts:
batch_1 = 100
batch_2 = 120
batch_3 = 80

Processed counts:
batch_1 = 100
batch_2 = 118
batch_3 = 80

Calculate the difference for each batch and identify which batch needs investigation


In [ ]:
expected = [100, 120, 80]
processed = [100, 118, 80]
 
for index in range(len(expected)):
    difference = expected[index] - processed[index]
 
    print(
        f"For batch {index + 1}: "
        f"difference = {difference}"
    )

    # batch_2 needs investigation because 2 records are missing from the processed count.


For batch 1: difference = 0
For batch 2: difference = 2
For batch 3: difference = 0


Q. 20 — Choose the Processing Style

For each case, choose batch or streaming and give one reason.
1. Store sends stock file every evening. - batch
2. Website sends click events all day. - streaming
3. Finance needs a report at the end of each month. - batch
4. Payment team needs alerts within a few seconds. - streaming
5. Supplier sends one price file every Monday. - batch
